<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/Regressie_analyse_G.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#All imports
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold



In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [ ]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [ ]:

#All the price features which include values as medium and medium_low
valuation_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Trailing P/E to Growth (PEG) ratio',
    'Book/Market',
    'Dividend Yield',
    'Dividend Payout Ratio'
]

# Select _Regime columns
regime_cols = [f"{col}_Regime" for col in valuation_features if f"{col}_Regime" in train.columns]

# Convert to string
for col in regime_cols:
    train[col] = train[col].astype(str)
    val[col] = val[col].astype(str)

# Missing values
for col in regime_cols:
    train[col] = train[col].replace("nan", "Missing")
    val[col] = val[col].replace("nan", "Missing")

# One-hot encoding (
train = pd.get_dummies(train, columns=regime_cols, drop_first=True)
val = pd.get_dummies(val, columns=regime_cols, drop_first=True)

# Align val and test on train columns
val = val.reindex(columns=train.columns, fill_value=0)


In [ ]:
#Making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_train = train["Return"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_val = val["Return"]


In [ ]:
#Dropping columns that do not matter
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])


In [ ]:
#making imputer
imputer = SimpleImputer(strategy='mean')

# Fit on train
X_train = imputer.fit_transform(X_train)

# Transform val
X_val = imputer.transform(X_val)


In [ ]:
#making a scaler
scaler = StandardScaler()

# Fit on train
X_train_scaled = scaler.fit_transform(X_train)

# Transform val
X_val_scaled = scaler.transform(X_val)

In [ ]:
# Making K-fold
X_all = np.vstack([X_train_scaled, X_val_scaled])
y_all = np.concatenate([y_train, y_val])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
r2_scores = []

In [ ]:
# Training model
for train_index, val_index in kf.split(X_all):
    X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
    y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

    # Model training
    model = LinearRegression()
    model.fit(X_train_fold, y_train_fold)

    # validate
    y_pred = model.predict(X_val_fold)

    # Saving scores
    mse_scores.append(mean_squared_error(y_val_fold, y_pred))
    r2_scores.append(r2_score(y_val_fold, y_pred))



In [ ]:
# Checking results
print("MSE per fold:", mse_scores)
print("Average MSE:", np.mean(mse_scores))
print("R² per fold:", r2_scores)
print("Average R²:", np.mean(r2_scores))

MSE per fold: [150.12971749055075, 132.58603055523614, 257.04857873795913, 160.25110088716795, 141.0450152050021]
Average MSE: 168.2120885751832
R² per fold: [-0.04317407886942792, -0.05672502574419891, -0.003414096782837772, -0.013287661431601094, -0.03633637651538746]
Average R²: -0.03058744786869063


In [ ]:
mse_scores = []
r2_scores = []
mae_scores = []

# Training model
for train_index, val_index in kf.split(X_all):
    X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
    y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

    model = LinearRegression()
    model.fit(X_train_fold, y_train_fold)

    y_pred = model.predict(X_val_fold)

    mse_scores.append(mean_squared_error(y_val_fold, y_pred))
    r2_scores.append(r2_score(y_val_fold, y_pred))

    mae_scores.append(mean_absolute_error(y_val_fold, y_pred))

rmse_scores = [np.sqrt(m) for m in mse_scores]

print("Average MAE:", np.mean(mae_scores))
print("Average RMSE:", np.mean(rmse_scores))

Average MAE: 4.76385389140416
Average RMSE: 12.86706968402089
